In [1]:
import pandas as pd
from src.utils.paths import load_paths

paths = load_paths()

vnat_feats = pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
iscx_feats = pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")

iscx_feats = iscx_feats[iscx_feats["q_min_packets_ok"] == 1.0].copy()

df_all = pd.concat([vnat_feats, iscx_feats], ignore_index=True)

# --- FIX: MERGE SPLITS FOR JOINT TRAINING ---
# We want to train on VNAT(train) + ISCX(train)
# We want to validate on VNAT(val) + ISCX(val)
# We keep ISCX(test) separate to see how it performs on held-out ISCX data.

# Map ISCX splits to standard names
df_all["split"] = df_all["split"].replace({
    "iscx_train": "train",
    "iscx_val": "val",
    # Keep iscx_test as iscx_test for separate evaluation,
    # BUT train_xgboost only reports metrics for 'test' split defined in yaml.
    # So let's map iscx_test to 'test' if we want it in the main report,
    # or keep it separate and evaluate manually.
    # Let's map it to 'test' so we get a combined test score,
    # and we can still filter by capture_id/dataset later if needed.
    "iscx_test": "test"
})

print("Splits after merging:")
print(df_all["split"].value_counts())
print("\nLabels:")
print(df_all["label"].value_counts())


Splits after merging:
split
train    11306
val       1132
test       551
Name: count, dtype: int64

Labels:
label
0    9672
1    3317
Name: count, dtype: int64


In [2]:
from src.models.xgb_train import train_xgboost
from src.utils.logging import setup_logger

logger = setup_logger(level="INFO")
xgb_yaml = paths.configs_dir / "xgb.yaml"

# Now this will train on BOTH datasets
res = train_xgboost(paths=paths, xgb_yaml=xgb_yaml, df=df_all)

print("Saved model:", res.model_path)
print("Saved metrics:", res.metrics_path)
print("Saved preds:", res.preds_path)

print("\nTrain (Combined):", res.metrics["splits"]["train"])
print("\nVal (Combined):", res.metrics["splits"]["val"])
print("\nTest (Combined):", res.metrics["splits"]["test"])
print("\nFirewall policy:", res.metrics.get("firewall_policy", res.metrics.get("policy_thresholds", {})))


DEBUG mean/std (first 5):
[-0.318 -0.753 -0.76  -0.7   -0.479]
[0.    0.029 0.061 0.055 0.   ]
[0]	train-logloss:0.64991	train-auc:0.98884	train-aucpr:0.95329	val-logloss:0.66241	val-auc:0.94024	val-aucpr:0.96730
[100]	train-logloss:0.01584	train-auc:0.99998	train-aucpr:0.99993	val-logloss:0.21646	val-auc:0.96609	val-aucpr:0.98307
[200]	train-logloss:0.00360	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.22807	val-auc:0.97301	val-aucpr:0.98648
[300]	train-logloss:0.00174	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.24598	val-auc:0.97366	val-aucpr:0.98687
[400]	train-logloss:0.00120	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.25749	val-auc:0.97374	val-aucpr:0.98692
[500]	train-logloss:0.00095	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.26668	val-auc:0.97413	val-aucpr:0.98706
[600]	train-logloss:0.00083	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.27406	val-auc:0.97432	val-aucpr:0.98705
[700]	train-logloss:0.00074	train-auc:1.00000	train-aucpr:1.00000	v

In [3]:
import json
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

from src.pipeline.artifacts import default_feature_artifacts
from src.pipeline.feature_pipeline import FeaturePipeline
from src.eval.metrics import pick_threshold_for_fpr, confusion_at_threshold

model_path = paths.repo_root / "artifacts" / "xgb" / "model.json"
booster = xgb.Booster()
booster.load_model(str(model_path))

feature_art = default_feature_artifacts(paths.artifacts_dir / "features")
pipeline = FeaturePipeline.load(feature_art)

X_all = pipeline.transform(df_all)
feat_cols = pipeline.model_feature_names()

# Evaluate specifically on ISCX Test subset (which is now part of 'test' split)
# We can identify it by capture_id or just filter the original dataframe logic if we had it.
# Since we merged splits, we can't easily distinguish VNAT test from ISCX test via 'split' col alone.
# But we can use the 'dataset' logic or capture_id patterns.

# Helper to identify ISCX rows
def is_iscx(row):
    # ISCX captures usually don't have the standard VNAT naming, or we can check capture_id
    # VNAT captures are like 'nonvpn_vimeo_audio_1'
    # ISCX captures are like 'vpn_skype_audio' (wait, they are similar)
    # But we know ISCX was appended last.
    # A better way is to reload ISCX feats to get the IDs.
    pass

# Reload ISCX test IDs
iscx_test_ids = set(pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")
                    [lambda d: d["split"] == "iscx_test"]["capture_id"])

iscx_test_mask = df_all["capture_id"].isin(iscx_test_ids)
iscx_test_df = df_all[iscx_test_mask].copy()

print(f"Recovered ISCX Test set: {len(iscx_test_df)} rows")

X_iscx = X_all.loc[iscx_test_df.index, feat_cols].to_numpy(dtype=float)
y_iscx = iscx_test_df["label"].to_numpy(dtype=int)

p_iscx = booster.predict(xgb.DMatrix(X_iscx, feature_names=feat_cols))

print("ISCX TEST ROC:", roc_auc_score(y_iscx, p_iscx))
print("ISCX TEST PR :", average_precision_score(y_iscx, p_iscx))

metrics_path = paths.repo_root / "artifacts" / "xgb" / "metrics.json"
m = json.loads(metrics_path.read_text(encoding="utf-8"))

# Use the firewall policy chosen during training
firewall_pol = m.get("firewall_policy", {})
thr = float(firewall_pol.get("threshold", 0.5))
pol_name = firewall_pol.get("chosen", "unknown")

print("Using firewall policy:", pol_name, "threshold:", thr)

yhat = (p_iscx >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_iscx, yhat).ravel()

prec = tp / (tp + fp + 1e-9)
rec  = tp / (tp + fn + 1e-9)
fpr  = fp / (fp + tn + 1e-9)

print("ISCX TEST @ firewall thr:", {"tn":tn,"fp":fp,"fn":fn,"tp":tp, "precision":prec, "recall":rec, "fpr":fpr})


Recovered ISCX Test set: 341 rows
ISCX TEST ROC: 0.8982214244921269
ISCX TEST PR : 0.9630142433170713
Using firewall policy: fpr_0_1pct threshold: 0.9838422536849976
ISCX TEST @ firewall thr: {'tn': np.int64(102), 'fp': np.int64(1), 'fn': np.int64(98), 'tp': np.int64(140), 'precision': np.float64(0.9929078014113979), 'recall': np.float64(0.5882352941151755), 'fpr': np.float64(0.00970873786398341)}


In [4]:
# Evaluate on VNAT Test subset
vnat_test_ids = set(pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
                    [lambda d: d["split"] == "test"]["capture_id"])

vnat_test_mask = df_all["capture_id"].isin(vnat_test_ids)
vnat_test_df = df_all[vnat_test_mask].copy()

print(f"Recovered VNAT Test set: {len(vnat_test_df)} rows")

X_vnat_test = X_all.loc[vnat_test_df.index, feat_cols].to_numpy(dtype=float)
y_vnat_test = vnat_test_df["label"].to_numpy(dtype=int)
p_vnat_test = booster.predict(xgb.DMatrix(X_vnat_test, feature_names=feat_cols))

yhat = (p_vnat_test >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_vnat_test, yhat).ravel()

prec = tp / (tp + fp + 1e-9)
rec  = tp / (tp + fn + 1e-9)
fpr  = fp / (fp + tn + 1e-9)

print("VNAT TEST @ firewall thr:", {"tn":tn,"fp":fp,"fn":fn,"tp":tp,
                                   "precision":prec,"recall":rec,"fpr":fpr})

Recovered VNAT Test set: 210 rows
VNAT TEST @ firewall thr: {'tn': np.int64(197), 'fp': np.int64(0), 'fn': np.int64(0), 'tp': np.int64(13), 'precision': np.float64(0.9999999999230769), 'recall': np.float64(0.9999999999230769), 'fpr': np.float64(0.0)}


In [5]:
def per_capture_recall(df_split, probs, thr):
    tmp = df_split[["capture_id", "label"]].copy()
    tmp["p"] = np.asarray(probs, dtype=float)
    tmp["yhat"] = (tmp["p"] >= float(thr)).astype(int)

    rows = []
    for cid, g in tmp.groupby("capture_id"):
        pos = g[g["label"] == 1]
        if len(pos) == 0:
            continue
        tp = int((pos["yhat"] == 1).sum())
        fn = int((pos["yhat"] == 0).sum())
        rec = tp / (tp + fn + 1e-9)
        rows.append((str(cid), rec, len(pos)))
    rows.sort(key=lambda x: x[1])
    return rows[:10]

worst_iscx = per_capture_recall(iscx_test_df, p_iscx, thr)
print("Worst 10 ISCX captures (capture_id, recall, #pos_flows):")
for cid, rec, npos in worst_iscx:
    print(cid, rec, npos)


Worst 10 ISCX captures (capture_id, recall, #pos_flows):
vpn_vpn_voipbuster1a.pcap 0.0 58
vpn_vpn_ftps_a.pcap 0.6696428571368782 112
vpn_vpn_hangouts_chat1b.pcap 0.9516129032104579 62
vpn_vpn_aim_chat1b.pcap 0.9999999998333333 6


In [6]:
# --- DIAGNOSE VOIPBUSTER ---
# Why is voipbuster failing?
vb_mask = iscx_test_df["capture_id"].str.contains("voipbuster")
vb_df = iscx_test_df[vb_mask].copy()

# Re-predict just for voipbuster to be safe
X_vb = X_all.loc[vb_df.index, feat_cols].to_numpy(dtype=float)
p_vb = booster.predict(xgb.DMatrix(X_vb, feature_names=feat_cols))

print(f"\nVoipbuster flows: {len(vb_df)}")
print(f"Mean Prob: {p_vb.mean():.4f} (Threshold: {thr:.4f})")
print(f"Max Prob:  {p_vb.max():.4f}")

# Check key features for voipbuster vs average VPN
print("\n--- Feature Comparison: Voipbuster vs All VPN ---")
vpn_mean = df_all[df_all["label"]==1][feat_cols].mean()
vb_mean = vb_df[feat_cols].mean()

# Show biggest differences (z-score style if we had std, but ratio works)
diffs = []
for c in feat_cols:
    # Use abs to avoid division by small numbers; add epsilon for safety
    if abs(vpn_mean[c]) > 1e-9:
        ratio = vb_mean[c] / (vpn_mean[c] + 1e-9)
        # Only show features that are significantly different
        if ratio < 0.5 or ratio > 2.0:
            diffs.append((c, ratio, vb_mean[c], vpn_mean[c]))

# Sort by the magnitude of the log of the ratio to find biggest proportional differences
diffs.sort(key=lambda x: abs(np.log(x[1] + 1e-9)), reverse=True)

print(f"{'Feature':<30} | {'Ratio':<6} | {'Voipbuster':<10} | {'Avg VPN':<10}")
for c, r, v, a in diffs[:15]:
    print(f"{c:<30} | {r:.2f}   | {v:10.4f} | {a:10.4f}")



Voipbuster flows: 58
Mean Prob: 0.1837 (Threshold: 0.9838)
Max Prob:  0.9809

--- Feature Comparison: Voipbuster vs All VPN ---
Feature                        | Ratio  | Voipbuster | Avg VPN   
f_total_pkts                   | -1.26   |    -0.2798 |     0.2227
f_up_pkts                      | -1.04   |    -0.2671 |     0.2575
f_down_pkts                    | -1.56   |    -0.2743 |     0.1763
f_total_bytes                  | -0.74   |    -0.1262 |     0.1708
f_up_bytes                     | -3.87   |    -0.1280 |     0.0331
f_down_bytes                   | -0.41   |    -0.0770 |     0.1893
f_iat_burstiness               | 7.45   |    -0.3739 |    -0.0502
f_bytes_per_s                  | 2.87   |    -0.6221 |    -0.2166
sz_all_count                   | -1.26   |    -0.2798 |     0.2227
sz_all_sum                     | -0.74   |    -0.1262 |     0.1708
sz_all_min                     | 15.41   |     1.8748 |     0.1217
f_pkts_per_s                   | 2.47   |    -0.6498 |    -0.2631
sz_a

C:\Users\scoti\AppData\Local\Temp\ipykernel_2140\1786443769.py:30: RuntimeWarning: invalid value encountered in log
  diffs.sort(key=lambda x: abs(np.log(x[1] + 1e-9)), reverse=True)


In [7]:
# --- CHECK TRAINING SET FOR VOIPBUSTER ---
train_df = df_all[df_all["split"] == "train"]
vb_train = train_df[train_df["capture_id"].str.contains("voipbuster")]
print(f"\nVoipbuster flows in TRAIN: {len(vb_train)}")
if len(vb_train) > 0:
    print("It IS in training. The model is just failing to learn it (likely due to feature scaling/outliers).")
else:
    print("It is NOT in training. This is a generalization failure.")



Voipbuster flows in TRAIN: 0
It is NOT in training. This is a generalization failure.
